# Portfolio Manager

Clean pipeline for mutual fund portfolio construction.

**Pipeline:**
1. Environment Setup
2. Paths & Parameters
3. Utility Functions
4. Core Logic (Filter → Rank → Overlap → Optimize)
5. Load & Validate Data
6. Run Pipeline
7. Results

## 1. Environment Setup

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

np.random.seed(42)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger(__name__)

## 2. Paths & Parameters

Edit this cell to change behavior without touching any logic.

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_FUNDS_PATH   = Path('../mutualfunds/raw_funds.tsv')
FUND_INFO_DIR    = Path('../mutualfunds/fund_info')
FUND_SCORES_PATH = Path('../mutualfunds/fund_scores.tsv')  # precomputed by fund_selection_strategy.ipynb
OUTPUT_DIR       = Path('tuning_results/portfolio_debug')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Portfolio Parameters ──────────────────────────────────────────────────────
NUM_FUNDS    = 8
RISK_PROFILE = 'aggressive'  # conservative | moderate | aggressive

# ── Scoring Weights (abs_score is pre-tested; adjust w_risk only) ─────────────
W_ABS  = 0.50   # weight on precomputed absolute quality score
W_RISK = 0.50   # weight on peer-relative risk score (Sharpe + Sortino)

# ── Overlap Constraint ────────────────────────────────────────────────────────
MAX_OVERLAP_PCT = 40.0   # max allowed pairwise portfolio overlap (%)

# ── Fund Universe ─────────────────────────────────────────────────────────────
FUND_NAMES = [
    'HDFC Mid Cap Dir Gr',
    'Edelweiss Mid Cap Dir Gr',
    'Nippon India Growth Mid Cap Dir Gr',
    'Invesco India Mid Cap Dir Gr',
    'ICICI Pru MidCap Dir Gr',
    'Nippon India Small Cap Dir Gr',
    'Quant Small Cap Dir Gr',
    'HDFC Small Cap Dir Gr',
    'Bandhan Small Cap Dir Gr',
    'HSBC Value Dir Gr',
    'Quant Flexi Cap Dir Gr',
    'HDFC Flexi Cap Dir Gr',
    'Parag Parikh Flexi Cap Dir Gr',
    'Bandhan Large & Mid Cap Dir Gr',
    'Motilal Oswal Large & Midcap Dir Gr',
    'ICICI Pru Large & Mid Cap Dir Gr'
]

# ── Validate prerequisites ────────────────────────────────────────────────────
assert RAW_FUNDS_PATH.exists(),   f'raw_funds.tsv not found at {RAW_FUNDS_PATH}'
assert FUND_INFO_DIR.exists(),    f'fund_info dir not found at {FUND_INFO_DIR}'
assert FUND_SCORES_PATH.exists(), (
    f'fund_scores.tsv not found at {FUND_SCORES_PATH}. '
    'Run fund_selection_strategy.ipynb first.'
)
assert RISK_PROFILE in ('conservative', 'moderate', 'aggressive'), \
    f'Invalid RISK_PROFILE: {RISK_PROFILE}'

logger.info(f'Parameters OK — {len(FUND_NAMES)} funds, profile={RISK_PROFILE}, '
            f'num_funds={NUM_FUNDS}, max_overlap={MAX_OVERLAP_PCT}%')

INFO — Parameters OK — 16 funds, profile=aggressive, num_funds=8, max_overlap=40.0%


## 3. Utility Functions

In [3]:
def safe_float(x):
    """
    Convert x to float, stripping commas. Returns None on failure.
    Callers must handle None explicitly — never silently default to 0.0.
    """
    try:
        return float(str(x).replace(',', ''))
    except (ValueError, TypeError):
        return None


def normalize_series(s: pd.Series) -> pd.Series:
    """
    Min-max normalize a Series to [0, 100].
    If all values are identical, returns 50.0 for all (neutral midpoint).
    """
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(50.0, index=s.index)
    return (s - mn) / (mx - mn) * 100.0


def load_risk_row(isin: str, fund_info_dir: Path):
    """
    Load the first row of risk_metrics_{isin}.tsv.
    Returns the row as a Series, or None if file is missing/empty.
    """
    risk_file = fund_info_dir / f'risk_metrics_{isin}.tsv'
    if not risk_file.exists():
        return None
    df = pd.read_csv(risk_file, sep='\t')
    if df.empty:
        return None
    return df.iloc[0]


logger.info('Utility functions defined.')

INFO — Utility functions defined.


## 4. Core Logic

### 4.1 Quality Filter

In [4]:
def quality_filter(fund_isins: list, fund_info_dir: Path) -> list:
    """
    Hard-gate filter. A fund must satisfy ALL of the following to pass:

      1. Sharpe_3Y  >= category average          (risk-adjusted return bar)
      2. Sortino_3Y >= category average          (downside risk bar; now required, not optional)
      3. Returns_3Y >= category average          (tightened from > 0; near-useless otherwise)
      4. StdDev_3Y  <= category average * 1.05   (tightened from 1.20; removes unjustified leniency)

    Funds with any missing required metric are skipped (not silently passed).

    Changes from v1:
      - Sortino promoted from optional to required
      - Returns threshold raised from > 0 to >= category average
      - StdDev buffer tightened from 1.20x to 1.05x
    """
    passed, skipped_missing, skipped_filter = [], [], []

    for isin in fund_isins:
        row = load_risk_row(isin, fund_info_dir)
        if row is None:
            skipped_missing.append(isin)
            continue

        sharpe        = safe_float(row.get('sharpe_3y'))
        sharpe_avg    = safe_float(row.get('sharpe_cat_avg_3y'))
        sortino       = safe_float(row.get('sortino_3y'))
        sortino_avg   = safe_float(row.get('sortino_cat_avg_3y'))
        returns       = safe_float(row.get('returns_3y'))
        returns_avg   = safe_float(row.get('returns_cat_avg_3y'))
        std           = safe_float(row.get('std_3y'))
        std_avg       = safe_float(row.get('std_cat_avg_3y'))

        # All metrics are now required
        required = [sharpe, sharpe_avg, sortino, sortino_avg,
                    returns, returns_avg, std, std_avg]
        if None in required:
            skipped_missing.append(isin)
            continue

        if (
            sharpe  >= sharpe_avg              and
            sortino >= sortino_avg             and
            returns >= returns_avg             and
            std     <= std_avg * 1.05
        ):
            passed.append(isin)
        else:
            skipped_filter.append(isin)

    logger.info(
        f'quality_filter: input={len(fund_isins)}, '
        f'passed={len(passed)}, '
        f'missing_data={len(skipped_missing)}, '
        f'below_bar={len(skipped_filter)}'
    )
    return passed


logger.info('quality_filter defined.')

INFO — quality_filter defined.


### 4.2 Ranking

In [5]:
def load_fund_scores(fund_scores_path: Path) -> dict:
    """
    Load precomputed absolute scores from fund_selection_strategy.ipynb.
    Returns dict keyed by ISIN.
    """
    df = pd.read_csv(fund_scores_path, sep='\t')
    return df.set_index('isin').to_dict(orient='index')


def rank_funds(filtered_isins: list, fund_info_dir: Path,
               fund_scores_path: Path,
               w_abs: float = W_ABS, w_risk: float = W_RISK) -> pd.DataFrame:
    """
    Hybrid scoring: absolute quality (w_abs) + peer-relative risk (w_risk).

    Absolute component (abs_score, 0-100):
        Precomputed by fund_selection_strategy.ipynb. Encodes long-term return
        percentile (10Y/5Y/3Y), stability CV, track-record tier, Crisil rating.
        Treated as ground truth — not recomputed here.

    Risk component (risk_score, 0-100):
        Sharpe and Sortino relative to category average only.
        Returns and StdDev are excluded — they are already encoded in Sharpe,
        and including them separately double-counts the signal.
        Formula: risk_ratio = 0.60 * (Sharpe/avg) + 0.40 * (Sortino/avg)
        Scaled: 1.0x category average → 50 pts.

    Blending:
        Both components are normalized to [0, 100] within the filtered universe
        BEFORE blending, so the w_abs / w_risk split is meaningful.

    Changes from v1/v2:
        - Removed Returns and StdDev from risk score (double-counting fix)
        - Normalized both components before blending (scale alignment fix)
    """
    abs_lookup = load_fund_scores(fund_scores_path)
    rows = []

    for isin in filtered_isins:
        row = load_risk_row(isin, fund_info_dir)
        if row is None:
            continue

        sharpe      = safe_float(row.get('sharpe_3y'))
        sharpe_avg  = safe_float(row.get('sharpe_cat_avg_3y'))
        sortino     = safe_float(row.get('sortino_3y'))
        sortino_avg = safe_float(row.get('sortino_cat_avg_3y'))

        # All four required (Sortino is now mandatory post-filter, but guard anyway)
        if None in [sharpe, sharpe_avg, sortino, sortino_avg]:
            continue
        if sharpe_avg == 0 or sortino_avg == 0:
            continue

        sharpe_r  = sharpe  / sharpe_avg
        sortino_r = sortino / sortino_avg

        # 0.0x avg → 0 pts | 1.0x avg → 50 pts | 2.0x avg → 100 pts
        risk_ratio = 0.60 * sharpe_r + 0.40 * sortino_r
        risk_score = min(100.0, max(0.0, risk_ratio * 50.0))

        info      = abs_lookup.get(isin, {})
        abs_score = info.get('abs_score', 50.0)  # fallback to neutral if missing

        rows.append({
            'ISIN':        isin,
            'abs_score':   round(abs_score, 2),
            'risk_score':  round(risk_score, 2),
            'sharpe_3y':   sharpe,
            'sortino_3y':  sortino,
            'tier':        info.get('tier', '?'),
            'CV':          info.get('CV'),
            's10Y':        info.get('s10Y'),
            's5Y':         info.get('s5Y'),
            's_stab':      info.get('s_stab'),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Normalize both components to [0, 100] within this universe before blending
    # This makes the w_abs / w_risk split genuinely 50/50 in contribution
    df['abs_norm']  = normalize_series(df['abs_score'])
    df['risk_norm'] = normalize_series(df['risk_score'])
    df['score']     = (w_abs * df['abs_norm'] + w_risk * df['risk_norm']).round(2)

    return df.sort_values('score', ascending=False).reset_index(drop=True)


logger.info('rank_funds defined.')

INFO — rank_funds defined.


### 4.3 Overlap Matrix

In [6]:
def compute_overlap_matrix(ranked_df: pd.DataFrame, fund_info_dir: Path) -> pd.DataFrame:
    """
    Compute pairwise holding overlap between funds (%).

    Overlap(A, B) = sum(min(w_a, w_b)) / avg(total_A, total_B) * 100

    Denominator uses the AVERAGE of both fund totals (Sørensen-style),
    not the minimum. This was the v1 bug: using min() biased overlap
    upward for small funds paired with large ones, incorrectly penalizing
    large-small combinations.

    Note: overlap is symmetric; diagonal is 100.0.
    """
    isins = ranked_df['ISIN'].tolist()
    holdings = {}
    missing = []

    for isin in isins:
        holdings_file = fund_info_dir / f'holdings_{isin}.tsv'
        if not holdings_file.exists():
            holdings[isin] = {}
            missing.append(isin)
            continue
        df = pd.read_csv(holdings_file, sep='\t')
        if df.empty:
            holdings[isin] = {}
            missing.append(isin)
            continue
        df['weight'] = df['weight'].apply(safe_float).fillna(0.0)
        df_agg = df.groupby('stock_name')['weight'].sum()
        holdings[isin] = df_agg.to_dict()

    if missing:
        logger.warning(f'Missing holdings for {len(missing)} funds: {missing}')

    n = len(isins)
    overlap = pd.DataFrame(0.0, index=isins, columns=isins)

    for i in range(n):
        for j in range(i, n):
            a, b = isins[i], isins[j]
            if i == j:
                overlap.loc[a, b] = 100.0
                continue

            stocks_a, stocks_b = holdings[a], holdings[b]
            all_stocks = set(stocks_a) | set(stocks_b)

            raw_overlap = sum(
                min(stocks_a.get(s, 0.0), stocks_b.get(s, 0.0))
                for s in all_stocks
            )

            total_a = sum(stocks_a.values())
            total_b = sum(stocks_b.values())
            denom   = (total_a + total_b) / 2.0   # Sørensen: average, not min

            pct = (raw_overlap / denom * 100.0) if denom > 0 else 0.0
            overlap.loc[a, b] = pct
            overlap.loc[b, a] = pct

    return overlap


logger.info('compute_overlap_matrix defined.')

INFO — compute_overlap_matrix defined.


### 4.4 Portfolio Optimizer

In [7]:
def optimize_portfolio(ranked_df: pd.DataFrame,
                       overlap_matrix: pd.DataFrame,
                       num_funds: int = NUM_FUNDS,
                       max_overlap_pct: float = MAX_OVERLAP_PCT) -> dict:
    """
    Combination-based portfolio optimizer.

    Selection:
        Evaluates all C(n, num_funds) combinations.
        Rejects any combo where any pairwise overlap > max_overlap_pct.
        Selects the combo with the highest total hybrid score.
        Overlap penalty (0.01 * sum) is intentionally removed — it was too
        small to affect selection at all and added false precision.

    Fallback:
        If NO combo passes the overlap filter, the threshold is relaxed in
        5% steps up to 2x max_overlap_pct and retried. This is logged loudly.
        Silent fallback to top-N (v1 behavior) is removed.

    Weight allocation:
        Proportional to hybrid score (same objective as selection — fixes v1
        mismatch where selection used hybrid score but weights used Sharpe/Std).
        Scores are floored at 0 before normalizing; if all are 0, equal weights.
    """
    isins     = ranked_df['ISIN'].tolist()
    score_map = dict(zip(ranked_df['ISIN'], ranked_df['score']))
    k         = min(num_funds, len(isins))

    def _find_best_combo(threshold):
        best_combo, best_score = None, -float('inf')
        for combo in combinations(isins, k):
            if any(
                overlap_matrix.loc[combo[i], combo[j]] > threshold
                for i in range(len(combo))
                for j in range(i + 1, len(combo))
            ):
                continue
            total = sum(score_map[f] for f in combo)
            if total > best_score:
                best_score, best_combo = total, combo
        return best_combo, best_score

    best_combo, best_score = _find_best_combo(max_overlap_pct)
    effective_threshold    = max_overlap_pct

    if best_combo is None:
        # Gradually relax threshold in 5% steps rather than silent top-N fallback
        for relaxed in range(int(max_overlap_pct) + 5,
                             int(max_overlap_pct * 2) + 1, 5):
            logger.warning(
                f'No combo passed overlap={effective_threshold:.0f}%. '
                f'Relaxing to {relaxed}%.'
            )
            best_combo, best_score = _find_best_combo(relaxed)
            effective_threshold    = relaxed
            if best_combo is not None:
                break

    if best_combo is None:
        raise ValueError(
            f'No valid portfolio found even at {effective_threshold:.0f}% overlap. '
            'Consider expanding the fund universe or raising MAX_OVERLAP_PCT.'
        )

    # Weights proportional to hybrid score
    raw = {isin: max(score_map.get(isin, 0.0), 0.0) for isin in best_combo}
    total = sum(raw.values())
    weights = (
        {k: v / total for k, v in raw.items()}
        if total > 0
        else {isin: 1.0 / len(best_combo) for isin in best_combo}
    )

    return {
        'selected_funds':       list(best_combo),
        'weights':              weights,
        'portfolio_score':      best_score,
        'effective_overlap_pct': effective_threshold,
    }


logger.info('optimize_portfolio defined.')

INFO — optimize_portfolio defined.


## 5. Load & Validate Data

In [8]:
# Load raw fund data
raw_df = pd.read_csv(RAW_FUNDS_PATH, sep='\t')
raw_df.columns = [c.strip() for c in raw_df.columns]

assert 'schemeName' in raw_df.columns, 'Missing schemeName column in raw_funds.tsv'
assert 'isin'       in raw_df.columns, 'Missing isin column in raw_funds.tsv'

ISIN_TO_NAME = dict(zip(raw_df['isin'], raw_df['schemeName']))

def get_fund_name(isin: str) -> str:
    return ISIN_TO_NAME.get(isin, f'Unknown ({isin})')

logger.info(f'Loaded {len(raw_df)} records from raw_funds.tsv')

INFO — Loaded 924 records from raw_funds.tsv


## 6. Resolve Fund Names → ISINs

In [9]:
def resolve_fund_names(names: list, raw_df: pd.DataFrame) -> pd.DataFrame:
    """Exact case-insensitive match of fund names to ISINs."""
    rows = []
    for name in names:
        match = raw_df[raw_df['schemeName'].str.lower() == name.lower()]
        isin  = str(match.iloc[0]['isin']) if not match.empty else None
        rows.append({'fund_name': name, 'isin': isin, 'resolved': isin is not None})
    return pd.DataFrame(rows)


resolve_df = resolve_fund_names(FUND_NAMES, raw_df)
unresolved = resolve_df[~resolve_df['resolved']]

if not unresolved.empty:
    logger.warning(f'{len(unresolved)} fund names could not be resolved:')
    print(unresolved[['fund_name']].to_string(index=False))

fund_isins = resolve_df.loc[resolve_df['resolved'], 'isin'].tolist()
assert fund_isins, 'No funds resolved. Check FUND_NAMES against raw_funds.tsv.'

logger.info(f'Resolved {len(fund_isins)}/{len(FUND_NAMES)} fund names to ISINs')

INFO — Resolved 16/16 fund names to ISINs


## 7. Run Pipeline

In [10]:
# ── Step 1: Quality Filter ────────────────────────────────────────────────────
print('=' * 70)
print('STEP 1 — Quality Filter')
print('=' * 70)

filtered_isins = quality_filter(fund_isins, FUND_INFO_DIR)
assert filtered_isins, (
    'No funds passed the quality filter. '
    'Consider relaxing StdDev buffer or checking data availability.'
)

print(f'\nPassed: {len(filtered_isins)} / {len(fund_isins)}')
for isin in filtered_isins:
    print(f'  {isin}  {get_fund_name(isin)}')

INFO — quality_filter: input=16, passed=10, missing_data=0, below_bar=6


STEP 1 — Quality Filter

Passed: 10 / 16
  INF179K01XQ0  HDFC Mid Cap Dir Gr
  INF843K01AO4  Edelweiss Mid Cap Dir Gr
  INF204K01E54  Nippon India Growth Mid Cap Dir Gr
  INF109K011N7  ICICI Pru MidCap Dir Gr
  INF204K01K15  Nippon India Small Cap Dir Gr
  INF917K01HD4  HSBC Value Dir Gr
  INF179K01UT0  HDFC Flexi Cap Dir Gr
  INF879O01027  Parag Parikh Flexi Cap Dir Gr
  INF194K01V89  Bandhan Large & Mid Cap Dir Gr
  INF109K011O5  ICICI Pru Large & Mid Cap Dir Gr


In [11]:
# ── Step 2: Rank Funds ────────────────────────────────────────────────────────
print('=' * 70)
print('STEP 2 — Ranking (hybrid score: abs + risk)')
print('=' * 70)

ranked_df = rank_funds(filtered_isins, FUND_INFO_DIR, FUND_SCORES_PATH,
                       w_abs=W_ABS, w_risk=W_RISK)
assert not ranked_df.empty, 'Ranking returned no results. Check risk_metrics files.'

ranked_df['name'] = ranked_df['ISIN'].map(get_fund_name)

display_cols = ['ISIN', 'score', 'abs_score', 'abs_norm',
                'risk_score', 'risk_norm', 'tier',
                's10Y', 's5Y', 's_stab', 'CV',
                'sharpe_3y', 'sortino_3y', 'name']

print(ranked_df[display_cols].to_string(index=True))

STEP 2 — Ranking (hybrid score: abs + risk)
           ISIN  score  abs_score    abs_norm  risk_score   risk_norm tier        s10Y         s5Y      s_stab         CV  sharpe_3y  sortino_3y                                name
0  INF879O01027  84.31      95.11   68.964260       99.86   99.652433    A   94.736842   87.500000   88.235294  70.342065       1.56        3.09       Parag Parikh Flexi Cap Dir Gr
1  INF179K01UT0  83.39      94.51   66.776076      100.00  100.000000    A   84.210526  100.000000   85.294118  71.467819       1.53        3.19               HDFC Flexi Cap Dir Gr
2  INF917K01HD4  53.19     103.62  100.000000       62.29    6.380338    A  100.000000  100.000000   94.736842  48.251177       1.24        2.08                   HSBC Value Dir Gr
3  INF194K01V89  51.48      96.03   72.319475       72.06   30.635551    A   95.000000   92.307692   84.615385  59.660606       1.32        2.29      Bandhan Large & Mid Cap Dir Gr
4  INF843K01AO4  41.18      95.16   69.146608      

In [12]:
# ── Step 3: Overlap Matrix ────────────────────────────────────────────────────
print('=' * 70)
print('STEP 3 — Overlap Matrix (Sørensen normalization)')
print('=' * 70)

overlap_df = compute_overlap_matrix(ranked_df, FUND_INFO_DIR)

# Display with fund names as labels for readability
name_map    = {isin: get_fund_name(isin)[:25] for isin in overlap_df.index}
overlap_named = overlap_df.rename(index=name_map, columns=name_map)
print(overlap_named.round(1).to_string())

# Flag high-overlap pairs for inspection
isins = overlap_df.index.tolist()
high_overlap = [
    (get_fund_name(isins[i])[:30], get_fund_name(isins[j])[:30],
     round(overlap_df.iloc[i, j], 1))
    for i in range(len(isins))
    for j in range(i + 1, len(isins))
    if overlap_df.iloc[i, j] > MAX_OVERLAP_PCT
]
if high_overlap:
    print(f'\n⚠️  Pairs above {MAX_OVERLAP_PCT}% overlap:')
    for a, b, pct in sorted(high_overlap, key=lambda x: -x[2]):
        print(f'  {pct:5.1f}%  {a}  ↔  {b}')
else:
    print(f'\n✓ No pairs exceed {MAX_OVERLAP_PCT}% overlap.')

STEP 3 — Overlap Matrix (Sørensen normalization)
                           Parag Parikh Flexi Cap Di  HDFC Flexi Cap Dir Gr  HSBC Value Dir Gr  Bandhan Large & Mid Cap D  Edelweiss Mid Cap Dir Gr  Nippon India Small Cap Di  HDFC Mid Cap Dir Gr  Nippon India Growth Mid C  ICICI Pru Large & Mid Cap  ICICI Pru MidCap Dir Gr
Parag Parikh Flexi Cap Di                      100.0                   34.9               19.4                       17.8                       2.9                        8.5                  2.3                        3.3                       18.7                      0.1
HDFC Flexi Cap Dir Gr                           34.9                  100.0               20.2                       26.6                       5.3                        9.7                 12.9                        9.4                       22.2                      2.7
HSBC Value Dir Gr                               19.4                   20.2              100.0                       24.8     

In [13]:
# ── Step 4: Portfolio Optimization ───────────────────────────────────────────
print('=' * 70)
print('STEP 4 — Portfolio Optimization')
print('=' * 70)

portfolio = optimize_portfolio(
    ranked_df, overlap_df,
    num_funds=NUM_FUNDS,
    max_overlap_pct=MAX_OVERLAP_PCT
)

if portfolio['effective_overlap_pct'] > MAX_OVERLAP_PCT:
    logger.warning(
        f"Overlap constraint relaxed to {portfolio['effective_overlap_pct']}% "
        f"(target was {MAX_OVERLAP_PCT}%). Consider expanding the fund universe."
    )

STEP 4 — Portfolio Optimization


## 8. Results

In [14]:
print('=' * 70)
print(f'PORTFOLIO  |  Profile: {RISK_PROFILE}  |  Funds: {NUM_FUNDS}')
print('=' * 70)
print(f'Portfolio Score      : {portfolio["portfolio_score"]:.2f}')
print(f'Overlap constraint   : {portfolio["effective_overlap_pct"]:.0f}%')
print()
print(f'{"Weight":>7}  {"Tier":>5}  {"Abs":>6}  {"Risk":>6}  {"Hybrid":>7}  Fund')
print('-' * 70)

for isin, weight in sorted(portfolio['weights'].items(), key=lambda x: -x[1]):
    row = ranked_df[ranked_df['ISIN'] == isin].iloc[0]
    print(
        f"{weight*100:6.1f}%  "
        f"{str(row['tier']):>5}  "
        f"{row['abs_score']:6.1f}  "
        f"{row['risk_score']:6.1f}  "
        f"{row['score']:7.2f}  "
        f"{get_fund_name(isin)}"
    )

print('-' * 70)
print(f'{'TOTAL':>7}  {sum(portfolio["weights"].values())*100:.1f}%')

# ── Save output ───────────────────────────────────────────────────────────────
result_rows = []
for isin, weight in portfolio['weights'].items():
    row = ranked_df[ranked_df['ISIN'] == isin].iloc[0]
    result_rows.append({
        'isin':       isin,
        'fund_name':  get_fund_name(isin),
        'weight_pct': round(weight * 100, 2),
        'tier':       row['tier'],
        'abs_score':  row['abs_score'],
        'risk_score': row['risk_score'],
        'hybrid':     row['score'],
        'sharpe_3y':  row['sharpe_3y'],
        'sortino_3y': row['sortino_3y'],
    })

result_df = pd.DataFrame(result_rows).sort_values('weight_pct', ascending=False)
out_path  = OUTPUT_DIR / 'portfolio.tsv'
result_df.to_csv(out_path, sep='\t', index=False)
logger.info(f'Portfolio saved to {out_path}')
print(f'\n✓ Saved to {out_path}')

INFO — Portfolio saved to tuning_results/portfolio_debug/portfolio.tsv


PORTFOLIO  |  Profile: aggressive  |  Funds: 8
Portfolio Score      : 424.30
Overlap constraint   : 40%

 Weight   Tier     Abs    Risk   Hybrid  Fund
----------------------------------------------------------------------
  19.9%      A    95.1    99.9    84.31  Parag Parikh Flexi Cap Dir Gr
  19.7%      A    94.5   100.0    83.39  HDFC Flexi Cap Dir Gr
  12.5%      A   103.6    62.3    53.19  HSBC Value Dir Gr
  12.1%      A    96.0    72.1    51.48  Bandhan Large & Mid Cap Dir Gr
   9.7%      A    95.2    65.0    41.18  Edelweiss Mid Cap Dir Gr
   9.2%      A    97.6    59.7    39.02  Nippon India Small Cap Dir Gr
   9.1%      A    89.9    70.9    38.76  HDFC Mid Cap Dir Gr
   7.8%      A    86.1    71.7    32.97  ICICI Pru Large & Mid Cap Dir Gr
----------------------------------------------------------------------
  TOTAL  100.0%

✓ Saved to tuning_results/portfolio_debug/portfolio.tsv
